# Chapter 11 — Classification

This notebook accompanies **Chapter 11** of *Inference in Statistical Modelling and Machine Learning*.

## Key ideas

Classification tasks predict a **categorical** outcome variable.  This chapter develops two probabilistic approaches:

* **Binary logistic regression** — models the log odds of the positive class as a linear function of predictors; fitted by minimising (regularised) cross-entropy loss.
* **Multinomial (softmax) logistic regression** — generalises to $K > 2$ classes using the softmax function.
* **$k$-nearest neighbours (k-NN)** — a non-parametric method that estimates class probabilities from the $k$ closest training points.

Throughout we use two datasets:

* **Diabetes dataset** (§11.1–11.3): 1 879 patients, 5 core predictors, binary outcome.
* **Human speech sounds** (§11.5–11.7): 2 371 vowel utterances, 10 syllable classes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.optimize import minimize

rng = np.random.default_rng(0)

In [ ]:
try:
    import google.colab
    DATA_PATH = "https://inferencebook.github.io/data/"
except ImportError:
    DATA_PATH = "../data/"


---
## §11.1 The diabetes dataset

> **Book link — §11.1.**  Table 11.1 lists five predictors tracked for each of the 1 879 patients; Table 11.2 shows the first few rows.

The response variable `Diagnosis` is 1 if the patient has diabetes, 0 otherwise.

In [ ]:
df = pd.read_csv(DATA_PATH + 'diabetes_data.csv')

FEATURE_COLS = ['FastingBloodSugar', 'HbA1c', 'CholesterolTriglycerides',
                'Hypertension', 'SocioeconomicStatus']
FEATURE_NAMES = ['Fasting blood sugar', 'Hemoglobin A1C',
                 'Cholesterol triglycerides', 'Hypertension', 'Socioeconomic status']
TARGET_COL = 'Diagnosis'

# 5-feature matrix and response
X5_raw = df[FEATURE_COLS].values.astype(float)
y      = df[TARGET_COL].values.astype(float)

n, p = X5_raw.shape
pi_hat = y.mean()   # base rate of diabetes

print(f"n = {n} patients,  {p} features")
print(f"Diabetes (y=1): {int(y.sum())}  ({100*pi_hat:.1f}%)")
print(f"Base-rate cross-entropy loss: {-(pi_hat*np.log(pi_hat)+(1-pi_hat)*np.log(1-pi_hat)):.4f}")
df[FEATURE_COLS + [TARGET_COL]].head()

---
## §11.2 Cross-entropy loss

> **Book link — §11.2 (eq. 11.3).**  For a probabilistic binary classifier $g$ and dataset $D$,  the cross-entropy loss is
> $$l(g; D) = -\frac{1}{n}\sum_{(x,y)\in D}\bigl(y\log g(x) + (1-y)\log(1-g(x))\bigr).$$
> A classifier that always predicts the base rate $\hat{\pi}$ achieves loss $-(\hat{\pi}\log\hat{\pi} + (1-\hat{\pi})\log(1-\hat{\pi}))$.  With $\hat{\pi}=0.4$ this equals **0.673** — the baseline to beat.

---
## §11.3.1 Binary logistic regression — the logit link

> **Book link — §11.3.1, Figure 11.1.**  The *logit* function transforms a probability $p\in(0,1)$ to the real line;
> its inverse — the *standard sigmoid* $\sigma(z)=(1+e^{-z})^{-1}$ — maps any real number back to $(0,1)$.
> $$\operatorname{logit}(p) = \log\frac{p}{1-p}, \qquad \sigma(z) = \frac{1}{1+e^{-z}}.$$

In [ ]:
# Figure 11.1 — logit and sigmoid
p_grid = np.linspace(0.001, 0.999, 500)
z_grid = np.linspace(-5, 5, 500)

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(9, 3.5))

ax_a.plot(p_grid, np.log(p_grid / (1 - p_grid)), 'k', lw=1.5)
ax_a.axhline(0, color='grey', lw=0.8, ls='--')
ax_a.set_xlabel('$p$'); ax_a.set_ylabel('logit$(p)$')
ax_a.set_title('(a) Logit function'); ax_a.set_xlim(0, 1)

ax_b.plot(z_grid, 1 / (1 + np.exp(-z_grid)), 'k', lw=1.5)
ax_b.axhline(0.5, color='grey', lw=0.8, ls='--')
ax_b.axvline(0,   color='grey', lw=0.8, ls='--')
ax_b.set_xlabel('$z$'); ax_b.set_ylabel(r'$\sigma(z)$')
ax_b.set_title('(b) Sigmoid function')

plt.suptitle('Figure 11.1 — logit and its inverse the sigmoid', y=1.01)
plt.tight_layout(); plt.show()

### Helper functions — binary logistic regression

We implement the loss function, its gradient, and fitting from scratch; `scipy.optimize.minimize`
with L-BFGS-B handles the actual optimisation.

In [ ]:
# ── Core helpers ──────────────────────────────────────────────────────────────

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def standardise(X_tr, X_te):
    # Standardise X_te using X_tr mean and std
    mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-15
    return (X_tr - mu) / sd, (X_te - mu) / sd

def bce_loss_and_grad(params, X, y, lam):
    # Regularised binary cross-entropy loss and gradient
    n = len(y)
    b0, beta = params[0], params[1:]
    p  = sigmoid(b0 + X @ beta)
    p  = np.clip(p, 1e-15, 1 - 1e-15)
    ce = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    loss = ce + lam / n * np.dot(beta, beta)
    err  = p - y
    grad_b0   = np.mean(err)
    grad_beta = X.T @ err / n + 2 * lam / n * beta
    return loss, np.concatenate([[grad_b0], grad_beta])

def fit_logreg(X_tr, y_tr, lam=0.0):
    # Fit logistic regression; return (b0, beta)
    p0 = np.zeros(X_tr.shape[1] + 1)
    res = minimize(bce_loss_and_grad, p0, jac=True, method='L-BFGS-B',
                   args=(X_tr, y_tr, lam), options={'maxiter': 500})
    return res.x[0], res.x[1:]   # (b0, beta)

def predict_proba(X, b0, beta):
    return sigmoid(b0 + X @ beta)

# ── Stratified 5-fold CV ───────────────────────────────────────────────────────
def stratified_kfold(y, k=5, seed=0):
    rng_ = np.random.default_rng(seed)
    pos = rng_.permutation(np.where(y == 1)[0])
    neg = rng_.permutation(np.where(y == 0)[0])
    return [np.concatenate([p, n])
            for p, n in zip(np.array_split(pos, k), np.array_split(neg, k))]

def cv_binary(X_raw, y, lam, k=5):
    # Return mean (ce_loss, accuracy) across k stratified folds
    folds = stratified_kfold(y, k)
    ces, accs = [], []
    for i in range(k):
        val = folds[i]
        tr  = np.concatenate([folds[j] for j in range(k) if j != i])
        X_tr_s, X_val_s = standardise(X_raw[tr], X_raw[val])
        b0, beta = fit_logreg(X_tr_s, y[tr], lam)
        pv = np.clip(predict_proba(X_val_s, b0, beta), 1e-15, 1 - 1e-15)
        ces.append(-np.mean(y[val]*np.log(pv) + (1-y[val])*np.log(1-pv)))
        accs.append(np.mean((pv > 0.5) == y[val]))
    return np.mean(ces), np.mean(accs)

print("Helper functions defined.")

---
## §11.3.2 Simple logistic regression on each predictor

> **Book link — §11.3.2, Figure 11.2.**  We fit five separate single-predictor models by
> maximising the unregularised likelihood ($\lambda=0$).  Each panel shows the fitted sigmoid
> together with jittered training data coloured by diagnosis (blue = diabetic, orange = healthy).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

b0s, betas = [], []

for j, (col, name) in enumerate(zip(FEATURE_COLS, FEATURE_NAMES)):
    xj = X5_raw[:, j:j+1]
    xj_s, _ = standardise(xj, xj)   # standardise using full dataset for plotting
    b0, beta = fit_logreg(xj_s, y, lam=0.0)
    b0s.append(b0); betas.append(beta[0])

    ax = axes[j]
    x_vals = np.linspace(xj[:, 0].min(), xj[:, 0].max(), 300)
    x_vals_s = (x_vals - xj[:, 0].mean()) / (xj[:, 0].std() + 1e-15)
    p_curve  = sigmoid(b0 + beta[0] * x_vals_s)

    ax.plot(x_vals, p_curve, 'steelblue', lw=2)

    jitter = rng.uniform(-0.02, 0.02, size=n)
    for cls, clr in [(1, 'steelblue'), (0, 'tomato')]:
        mask = (y == cls)
        ax.scatter(xj[mask, 0], jitter[mask], c=clr, s=6, alpha=0.5,
                   label=['No diabetes', 'Diabetes'][cls])

    ax.set_xlabel(name); ax.set_ylabel('P(diabetes)')
    ax.set_ylim(-0.08, 1.08)

axes[-1].axis('off')
axes[0].legend(fontsize=8, loc='upper left')
plt.suptitle('Figure 11.2 — Simple logistic regression on each of the five predictors', y=1.01)
plt.tight_layout(); plt.show()

### Table 11.3 — Coefficients, bootstrap SEs, and CV performance

> **Book link — §11.3.2, Table 11.3.**  Standard errors use 1 000 non-parametric bootstrap
> resamples.  Starred coefficients ($\hat{\beta}_1^*$) are based on standardised predictors.
> CV scores use stratified 5-fold cross-validation.

*This cell may take 30–60 s to run.*

In [ ]:
N_BOOT = 1000
boot_betas = np.zeros((N_BOOT, len(FEATURE_COLS)))

for b in range(N_BOOT):
    idx = rng.integers(0, n, size=n)
    for j, col in enumerate(FEATURE_COLS):
        xj = X5_raw[idx, j:j+1]
        xj_s, _ = standardise(xj, xj)
        _, beta_b = fit_logreg(xj_s, y[idx], lam=0.0)
        boot_betas[b, j] = beta_b[0]

se_boot = boot_betas.std(axis=0)

# CV scores
cv_ce  = np.zeros(len(FEATURE_COLS))
cv_acc = np.zeros(len(FEATURE_COLS))
for j, col in enumerate(FEATURE_COLS):
    cv_ce[j], cv_acc[j] = cv_binary(X5_raw[:, j:j+1], y, lam=0.0)

# Raw (unstandardised) coefficients
raw_betas = np.array(betas)
raw_b0s   = np.array(b0s)

print(f"{'Predictor':<30} {'β₁':>8} {'ŝe(β₁)':>10} {'β₁*':>8} {'ŝe(β₁*)':>10} {'CV CE':>8} {'CV acc':>8}")
print("-" * 84)
for j, name in enumerate(FEATURE_NAMES):
    # raw β₁ (for standardised predictor that's what we fit)
    xj = X5_raw[:, j:j+1]
    mu_j, sd_j = xj.mean(), xj.std()
    b1_star = raw_betas[j]           # coefficient on standardised predictor
    b1_raw  = b1_star / sd_j         # coefficient on original scale
    print(f"{name:<30} {b1_raw:>8.4f} {'—':>10} {b1_star:>8.4f} {se_boot[j]:>10.4f}"
          f" {cv_ce[j]:>8.3f} {cv_acc[j]:>8.3f}")
print()
print("(Table 11.3 target values:  FBS β*=1.080 se=0.059  CV acc=0.680)")
print("                            HbA1c β*=1.012 se=0.057  CV acc=0.668")

---
## §11.3.3 Why regularisation is needed — linear separability

> **Book link — §11.3.3, Figure 11.3.**  As the two classes become more distinct, the
> unregularised MLE sigmoid becomes sharper.  When classes are *linearly separable* (panel d)
> there is no MLE: the loss keeps falling as $|\beta_1| \to \infty$.

In [ ]:
# Simulate four datasets with increasing separation
datasets = [
    (rng.normal(-0.8, 1.2, 100), rng.normal(0.8, 1.2, 100)),   # (a) overlapping
    (rng.normal(-1.5, 0.9, 100), rng.normal(1.5, 0.9, 100)),   # (b) more distinct
    (rng.normal(-2.5, 0.7, 100), rng.normal(2.5, 0.7, 100)),   # (c) quite distinct
    (rng.normal(-3.5, 0.3, 100), rng.normal(3.5, 0.3, 100)),   # (d) linearly separable
]
lams_fig3 = [0.0, 0.0, 0.0, 0.5]   # small λ for (d) so optimisation converges

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()
titles = ['(a)', '(b)', '(c)', '(d)']

for ax, (neg_x, pos_x), lam_d, title in zip(axes, datasets, lams_fig3, titles):
    X_d = np.concatenate([neg_x, pos_x]).reshape(-1, 1)
    y_d = np.array([0]*100 + [1]*100, dtype=float)
    X_d_s, _ = standardise(X_d, X_d)

    b0_d, beta_d = fit_logreg(X_d_s, y_d, lam=lam_d)

    z_range = np.linspace(X_d.min() - 0.5, X_d.max() + 0.5, 400)
    z_s = (z_range - X_d[:, 0].mean()) / (X_d[:, 0].std() + 1e-15)
    p_curve = sigmoid(b0_d + beta_d[0] * z_s)

    jitter_y = rng.uniform(-0.04, 0.04, size=200)
    ax.plot(z_range, p_curve, 'steelblue', lw=2)
    ax.scatter(neg_x, jitter_y[:100], c='tomato',    s=15, alpha=0.7)
    ax.scatter(pos_x, jitter_y[100:], c='steelblue', s=15, alpha=0.7)
    ax.set_title(title); ax.set_xlabel('$x$'); ax.set_ylabel(r'$p_{\rm blue}$')
    ax.set_ylim(-0.08, 1.08)
    if title == '(d)':
        ax.text(0.05, 0.92, f'λ={lam_d} (reg. needed)', transform=ax.transAxes,
                fontsize=8, color='darkred')

plt.suptitle('Figure 11.3 — Logistic regression with increasing class separation', y=1.01)
plt.tight_layout(); plt.show()

---
## §11.3.5 Multiple logistic regression on the diabetes dataset

> **Book link — §11.3.5, Box 11.1, Figure 11.4.**  All five predictors are standardised jointly
> and a ridge penalty $\lambda$ is tuned by 5-fold CV.  Panel (c)/(d) zoom in on the
> optimal region.

In [ ]:
# Coarse scan: λ ∈ [0, 2000]
lams_coarse = np.concatenate([[0], np.logspace(-1, 3.3, 60)])

ce_cv_coarse  = np.zeros(len(lams_coarse))
acc_cv_coarse = np.zeros(len(lams_coarse))
for i, lam in enumerate(lams_coarse):
    ce_cv_coarse[i], acc_cv_coarse[i] = cv_binary(X5_raw, y, lam)
    if i % 10 == 0:
        print(f"  λ={lam:.2f}  CE={ce_cv_coarse[i]:.4f}  acc={acc_cv_coarse[i]:.4f}")

# Fine scan: λ ∈ [0, 4]
lams_fine = np.linspace(0, 4, 80)
ce_cv_fine  = np.zeros(len(lams_fine))
acc_cv_fine = np.zeros(len(lams_fine))
for i, lam in enumerate(lams_fine):
    ce_cv_fine[i], acc_cv_fine[i] = cv_binary(X5_raw, y, lam)

best_lam_ce  = lams_fine[ce_cv_fine.argmin()]
best_lam_acc = lams_fine[acc_cv_fine.argmax()]
print(f"\nBest λ for CE loss : {best_lam_ce:.2f}")
print(f"Best λ for accuracy: {best_lam_acc:.2f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
((ax_a, ax_b), (ax_c, ax_d)) = axes

ax_a.plot(lams_coarse, ce_cv_coarse, 'steelblue', lw=1.5)
ax_a.set_xlabel('λ'); ax_a.set_ylabel('CV CE loss'); ax_a.set_title('(a)')

ax_b.plot(lams_coarse, acc_cv_coarse, 'steelblue', lw=1.5)
ax_b.set_xlabel('λ'); ax_b.set_ylabel('CV accuracy'); ax_b.set_title('(b)')

ax_c.plot(lams_fine, ce_cv_fine, 'steelblue', lw=1.5)
ax_c.axvline(best_lam_ce, color='tomato', ls='--', lw=1, label=f'λ*={best_lam_ce:.2f}')
ax_c.set_xlabel('λ'); ax_c.set_ylabel('CV CE loss'); ax_c.set_title('(c) zoomed'); ax_c.legend()

ax_d.plot(lams_fine, acc_cv_fine, 'steelblue', lw=1.5)
ax_d.axvline(best_lam_acc, color='tomato', ls='--', lw=1, label=f'λ*={best_lam_acc:.2f}')
ax_d.set_xlabel('λ'); ax_d.set_ylabel('CV accuracy'); ax_d.set_title('(d) zoomed'); ax_d.legend()

plt.suptitle('Figure 11.4 — CV performance of multiple logistic regression (5 predictors)', y=1.01)
plt.tight_layout(); plt.show()

### Table 11.4 — Fitted coefficients at λ = 0.7

> **Book link — §11.3.5, Table 11.4.**  Fit on the full dataset with $\lambda = 0.7$.
> Bootstrap SEs use 1 000 resamples.

In [ ]:
LAM_FINAL = 0.7
X5_s, _ = standardise(X5_raw, X5_raw)   # standardise using full dataset
b0_full, beta_full = fit_logreg(X5_s, y, lam=LAM_FINAL)

# Bootstrap SEs for the 5-predictor model
N_BOOT2 = 1000
boot5 = np.zeros((N_BOOT2, len(FEATURE_COLS)))
for b in range(N_BOOT2):
    idx = rng.integers(0, n, size=n)
    Xb, yb = X5_raw[idx], y[idx]
    Xb_s, _ = standardise(Xb, Xb)
    _, beta_b = fit_logreg(Xb_s, yb, lam=LAM_FINAL)
    boot5[b] = beta_b
se5 = boot5.std(axis=0)

print(f"λ = {LAM_FINAL}")
print(f"{'Predictor':<30} {'β̂':>8} {'ŝe(β̂)':>10}")
print("-" * 52)
for j, name in enumerate(FEATURE_NAMES):
    print(f"{name:<30} {beta_full[j]:>8.3f} {se5[j]:>10.3f}")

cv_ce_07, cv_acc_07 = cv_binary(X5_raw, y, lam=LAM_FINAL)
print(f"\nCV CE loss = {cv_ce_07:.4f}   CV accuracy = {cv_acc_07:.4f}")
print("(Book: β_FBS=1.584, β_HbA1c=1.494, CV accuracy ≈ 0.826)")

### Figure 11.5 — 43-feature dataset (n = 300)

> **Book link — §11.3.5, Figure 11.5.**  The diabetes dataset originally has 43 features.
> We sample 300 of the 1 879 examples and apply the same CV workflow.
> With fewer examples and many more features, regularisation helps much more.

In [ ]:
EXCLUDE_COLS = ['PatientID', 'DoctorInCharge', TARGET_COL]
ALL_FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]
print(f"All features: {len(ALL_FEATURE_COLS)}")

# Sample 300 rows
idx300 = rng.choice(n, size=300, replace=False)
X43_raw = df[ALL_FEATURE_COLS].values.astype(float)[idx300]
y300    = y[idx300]
print(f"Wide dataset: {X43_raw.shape},  diabetes rate = {y300.mean():.3f}")

lams_wide  = np.concatenate([[0], np.logspace(-0.5, 2.5, 50)])
ce_wide  = np.zeros(len(lams_wide))
acc_wide = np.zeros(len(lams_wide))
for i, lam in enumerate(lams_wide):
    ce_wide[i], acc_wide[i] = cv_binary(X43_raw, y300, lam)

# Fine zoom 0–16
lams_wide_f = np.linspace(0, 16, 80)
ce_wide_f  = np.zeros(len(lams_wide_f))
acc_wide_f = np.zeros(len(lams_wide_f))
for i, lam in enumerate(lams_wide_f):
    ce_wide_f[i], acc_wide_f[i] = cv_binary(X43_raw, y300, lam)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
((ax_a, ax_b), (ax_c, ax_d)) = axes

ax_a.plot(lams_wide, ce_wide, 'steelblue', lw=1.5)
ax_a.set_xlabel('λ'); ax_a.set_ylabel('CV CE loss'); ax_a.set_title('(a) 43 features, n=300')

ax_b.plot(lams_wide, acc_wide, 'steelblue', lw=1.5)
ax_b.set_xlabel('λ'); ax_b.set_ylabel('CV accuracy'); ax_b.set_title('(b)')

ax_c.plot(lams_wide_f, ce_wide_f, 'steelblue', lw=1.5)
ax_c.set_xlabel('λ'); ax_c.set_ylabel('CV CE loss'); ax_c.set_title('(c) zoomed')

ax_d.plot(lams_wide_f, acc_wide_f, 'steelblue', lw=1.5)
ax_d.set_xlabel('λ'); ax_d.set_ylabel('CV accuracy'); ax_d.set_title('(d) zoomed')

plt.suptitle('Figure 11.5 — CV for 43-feature logistic regression (n=300)', y=1.01)
plt.tight_layout(); plt.show()

---
## §11.5 Multinomial logistic regression — human speech sounds

> **Book link — §11.5.2, Figure 11.6.**  We distinguish 10 English vowel syllables (K=10)
> using formant measurements: **backness** ($X_1$) and **height** ($X_2$).
> Data: Clopper et al. (2005), 48 speakers, n=2 371 utterances.

In [ ]:
vdf = pd.read_csv(DATA_PATH + 'Vowels_Processed.csv')

# 10 standard h_Vd frame vowels (filter out 'frogs' and 'logs')
HVD_CLASSES = ['had', 'hayed', 'head', 'heed', 'hid', 'hod', 'hoed', 'hood', 'hud', 'whod']
vdf = vdf[vdf['Target'].isin(HVD_CLASSES)].copy().reset_index(drop=True)
print(f"Vowel dataset: {len(vdf)} utterances, {vdf['Target'].nunique()} classes")

# Integer-encode the target
vdf['class_id'] = vdf['Target'].map({v: i for i, v in enumerate(HVD_CLASSES)})
Xv   = vdf[['Backness', 'Height']].values.astype(float)   # (2371, 2)
yv   = vdf['class_id'].values                              # (2371,)
Kv   = len(HVD_CLASSES)
COLORS_10 = [f'C{i}' for i in range(10)]

# Figure 11.6 — scatter plot
fig, ax = plt.subplots(figsize=(7, 6))
for k, name in enumerate(HVD_CLASSES):
    mask = (yv == k)
    ax.scatter(Xv[mask, 0], Xv[mask, 1], c=COLORS_10[k], s=8, alpha=0.7,
               label=f'{k}: {name}')
ax.set_xlabel('$X_1$ (Backness)'); ax.set_ylabel('$X_2$ (Height)')
ax.set_title('Figure 11.6 — Vowel backness and height for 10 syllables')
ax.legend(fontsize=7, markerscale=2, loc='upper right')
plt.tight_layout(); plt.show()

### Multinomial logistic regression — from-scratch implementation

> **Book link — §11.5.1, Box 11.2 (eqs 11.17–11.19).**
> Score vector $\mathbf{s} = \boldsymbol{\beta}_0 + \underline{\boldsymbol{\beta}}\mathbf{x}$,
> class probabilities $\mathbf{g} = \operatorname{softmax}(\mathbf{s})$,
> fitted by minimising regularised categorical cross-entropy loss.

In [ ]:
def softmax(S):
    # Numerically stable softmax; S shape (n, K)
    S = S - S.max(axis=1, keepdims=True)
    E = np.exp(S)
    return E / E.sum(axis=1, keepdims=True)

def cce_loss_and_grad(params_flat, X, Y_oh, lam):
    # Categorical cross-entropy + ridge; Y_oh is one-hot (n, K)
    n, p = X.shape
    K = Y_oh.shape[1]
    b0 = params_flat[:K]
    B  = params_flat[K:].reshape(K, p)   # (K, p) weight matrix
    S  = X @ B.T + b0                    # (n, K) scores
    P  = np.clip(softmax(S), 1e-15, 1.0)
    loss = -np.mean(np.sum(Y_oh * np.log(P), axis=1)) + lam / n * np.sum(B**2)
    dS  = (P - Y_oh) / n                 # (n, K)
    db0 = dS.sum(axis=0)                 # (K,)
    dB  = dS.T @ X + 2 * lam / n * B    # (K, p)
    return loss, np.concatenate([db0, dB.ravel()])

def one_hot(y, K):
    Y = np.zeros((len(y), K))
    Y[np.arange(len(y)), y] = 1.0
    return Y

def fit_softmax(X_tr, y_tr, K, lam=0.0):
    # Return (b0 shape (K,), B shape (K,p))
    p   = X_tr.shape[1]
    Y_oh = one_hot(y_tr, K)
    res = minimize(cce_loss_and_grad, x0=np.zeros(K + K*p), jac=True,
                   method='L-BFGS-B', args=(X_tr, Y_oh, lam),
                   options={'maxiter': 1000})
    return res.x[:K], res.x[K:].reshape(K, p)

def predict_proba_softmax(X, b0, B):
    return softmax(X @ B.T + b0)

def cv_softmax(X_raw, y, K, lam, k=5):
    folds = stratified_kfold(y, k)
    ces, accs = [], []
    for i in range(k):
        val = folds[i]
        tr  = np.concatenate([folds[j] for j in range(k) if j != i])
        X_tr_s, X_val_s = standardise(X_raw[tr], X_raw[val])
        b0, B = fit_softmax(X_tr_s, y[tr], K, lam)
        P_val = np.clip(predict_proba_softmax(X_val_s, b0, B), 1e-15, 1)
        ce  = -np.mean(np.log(P_val[np.arange(len(val)), y[val]]))
        acc = np.mean(P_val.argmax(1) == y[val])
        ces.append(ce); accs.append(acc)
    return np.mean(ces), np.mean(accs)

print("Multinomial LR helpers defined.")

### Figure 11.7 — Cross-validation for multinomial logistic regression on vowels

> **Book link — §11.5.2, Figure 11.7.**  Very light regularisation ($\lambda \approx 0.002$)
> is optimal.  The dataset is tall ($n=2371$) and the feature space small ($p=2$),
> so overfitting risk is low.

In [ ]:
lams_v = np.linspace(0, 0.01, 50)
ce_v   = np.zeros(len(lams_v))
acc_v  = np.zeros(len(lams_v))

for i, lam in enumerate(lams_v):
    ce_v[i], acc_v[i] = cv_softmax(Xv, yv, Kv, lam)
    if i % 10 == 0:
        print(f"  λ={lam:.4f}  CE={ce_v[i]:.4f}  acc={acc_v[i]:.4f}")

best_lam_v = lams_v[ce_v.argmin()]
print(f"\nBest λ (CE loss) = {best_lam_v:.4f}")
print(f"Best CV accuracy  = {acc_v.max():.4f}")
print("(Book: λ ≈ 0.002, accuracy ≈ 0.889)")

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(10, 4))
ax_a.plot(lams_v, ce_v, 'steelblue', lw=1.5)
ax_a.axvline(best_lam_v, color='tomato', ls='--', lw=1)
ax_a.set_xlabel('λ'); ax_a.set_ylabel('CV CCE loss'); ax_a.set_title('(a)')

ax_b.plot(lams_v, acc_v, 'steelblue', lw=1.5)
ax_b.axvline(best_lam_v, color='tomato', ls='--', lw=1)
ax_b.set_xlabel('λ'); ax_b.set_ylabel('CV accuracy'); ax_b.set_title('(b)')

plt.suptitle('Figure 11.7 — CV for multinomial logistic regression on speech sounds', y=1.01)
plt.tight_layout(); plt.show()

### Figure 11.8 — Decision regions of the fitted multinomial logistic regression model

> **Book link — §11.5.2, Figure 11.8.**  Background colour = best-guess class label;
> opacity = classifier confidence.  Boundaries between decision regions are straight lines
> (because scores are linear in $\mathbf{x}$).

In [ ]:
LAM_V = 0.002   # chosen by CV

# Fit on full dataset
Xv_s, _ = standardise(Xv, Xv)
b0_v, B_v = fit_softmax(Xv_s, yv, Kv, lam=LAM_V)

# Decision region helper
def make_background(ax, proba_fn, xlim, ylim, colors, ngrid=250):
    x1 = np.linspace(*xlim, ngrid)
    x2 = np.linspace(*ylim, ngrid)
    xx, yy = np.meshgrid(x1, x2)
    grid = np.c_[xx.ravel(), yy.ravel()]
    P   = proba_fn(grid)                 # (ngrid^2, K)
    best = P.argmax(1)
    conf = P.max(1)
    rgba = np.zeros((ngrid * ngrid, 4))
    for k, c in enumerate(colors):
        m = (best == k)
        rgba[m, :3] = np.array(mcolors.to_rgb(c))
        rgba[m, 3]  = conf[m]
    ax.imshow(rgba.reshape(ngrid, ngrid, 4),
              extent=[*xlim, *ylim], origin='lower', aspect='auto', interpolation='bilinear')

# Standardise grid using full-data stats
mu_v, sd_v = Xv.mean(0), Xv.std(0)
def predict_grid_softmax(grid):
    return predict_proba_softmax((grid - mu_v) / sd_v, b0_v, B_v)

XLIM = (-3.0, 2.5); YLIM = (-3.0, 2.5)

fig, ax = plt.subplots(figsize=(7, 6))
make_background(ax, predict_grid_softmax, XLIM, YLIM, COLORS_10)
for k, name in enumerate(HVD_CLASSES):
    mask = (yv == k)
    ax.scatter(Xv[mask, 0], Xv[mask, 1], c=COLORS_10[k], s=6, alpha=0.8, linewidths=0)
ax.set_xlim(XLIM); ax.set_ylim(YLIM)
ax.set_xlabel('$X_1$ (Backness)'); ax.set_ylabel('$X_2$ (Height)')
ax.set_title(f'Figure 11.8 — MLR decision regions (λ={LAM_V})')
plt.tight_layout(); plt.show()

---
## §11.6 $k$-nearest neighbours

> **Book link — §11.6.1.**  k-NN is a non-parametric classifier.  For an unlabelled point $\mathbf{x}$,
> the predicted class probabilities are the empirical distribution of class labels among the
> $k$ nearest training points (by Euclidean distance after standardisation).

> **Book link — §11.6.2, Figure 11.9.**  Cross-validated accuracy peaks around $k \approx 18$.

In [ ]:
# ── k-NN helpers ──────────────────────────────────────────────────────────────

def knn_proba(X_tr, y_tr, X_te, k, K):
    # Vectorised k-NN probability estimates
    D = np.sum((X_te[:, None, :] - X_tr[None, :, :])**2, axis=2)   # (n_te, n_tr)
    nn = np.argsort(D, axis=1)[:, :k]                                # (n_te, k)
    labs = y_tr[nn]                                                   # (n_te, k)
    P = np.zeros((len(X_te), K))
    for c in range(K):
        P[:, c] = (labs == c).mean(axis=1)
    return P

def cv_knn_scan(X_raw, y, K, k_values, n_folds=5):
    # CV accuracy for a range of k values; compute distance matrix per fold
    folds = stratified_kfold(y, n_folds)
    acc_matrix = np.zeros((n_folds, len(k_values)))
    for fi in range(n_folds):
        val = folds[fi]
        tr  = np.concatenate([folds[j] for j in range(n_folds) if j != fi])
        X_tr_s, X_val_s = standardise(X_raw[tr], X_raw[val])
        # Precompute distance matrix and argsort once
        D   = np.sum((X_val_s[:, None, :] - X_tr_s[None, :, :])**2, axis=2)
        srt = np.argsort(D, axis=1)          # sorted neighbour indices
        for ki, k in enumerate(k_values):
            labs = y[tr][srt[:, :k]]
            pred = np.array([np.bincount(row, minlength=K).argmax() for row in labs])
            acc_matrix[fi, ki] = (pred == y[val]).mean()
    return acc_matrix.mean(axis=0)

K_VALUES = np.arange(1, 301)
print("Running k-NN CV scan (k=1..300)…")
acc_knn_cv = cv_knn_scan(Xv, yv, Kv, K_VALUES)
best_k = K_VALUES[acc_knn_cv.argmax()]
print(f"Best k = {best_k},  CV accuracy = {acc_knn_cv.max():.4f}")
print("(Book: k ≈ 18, accuracy ≈ 0.90)")

In [ ]:
# Figure 11.9(a) — CV accuracy vs k
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
ax_a = axes[0, 0]
ax_a.plot(K_VALUES, acc_knn_cv, 'steelblue', lw=1)
ax_a.axvline(best_k, color='tomato', ls='--', lw=1, label=f'k*={best_k}')
ax_a.set_xlabel('$k$'); ax_a.set_ylabel('CV accuracy')
ax_a.set_title('(a) CV accuracy vs k'); ax_a.legend()

# Decision regions for k = 3, 10, 100
Xv_s_full, _ = standardise(Xv, Xv)

for ax, k_val, label in zip([axes[0,1], axes[1,0], axes[1,1]],
                              [3, 10, 100], ['(b) k=3', '(c) k=10', '(d) k=100']):
    def make_pred(grid, k=k_val):
        grid_s = (grid - mu_v) / sd_v
        return knn_proba(Xv_s_full, yv, grid_s, k, Kv)

    make_background(ax, make_pred, XLIM, YLIM, COLORS_10, ngrid=150)
    ax.set_xlim(XLIM); ax.set_ylim(YLIM)
    ax.set_xlabel('$X_1$'); ax.set_ylabel('$X_2$')
    ax.set_title(label)

plt.suptitle('Figure 11.9 — k-NN on vowel dataset', y=1.01)
plt.tight_layout(); plt.show()

---
## §11.7 Multinomial logistic regression vs k-NN in higher dimensions

> **Book link — §11.7, Table 11.5.**  We restore two additional feature groups:
> **duration** (one continuous feature) and **dialect** (six one-hot columns for
> Mid-Atlantic, Midland, New England, North, South, West),
> giving a 9-dimensional predictor space.
>
> Parametric methods (MLR) tend to outperform distance-based ones (k-NN) as $p$ grows.

In [ ]:
# Build 9-feature dataset
DIALECT_LABELS = ['Mid-Atlantic', 'Midland', 'NewEngland', 'North', 'South', 'West']
dial_oh = np.zeros((len(vdf), len(DIALECT_LABELS)))
for j, d in enumerate(DIALECT_LABELS):
    dial_oh[:, j] = (vdf['Dialect'] == d).astype(float)

Xv9 = np.column_stack([
    Xv,                        # Backness, Height
    vdf['duration'].values,    # duration
    dial_oh                    # 6 one-hot dialect columns
])
print(f"Extended feature matrix: {Xv9.shape}")

# CV for multinomial LR on 9 features
lams_v9 = np.linspace(0, 0.05, 30)
acc_v9  = [cv_softmax(Xv9, yv, Kv, lam)[1] for lam in lams_v9]
best_lam_v9 = lams_v9[np.argmax(acc_v9)]
ce_v9_best, acc_v9_best = cv_softmax(Xv9, yv, Kv, lam=best_lam_v9)
print(f"MLR (9 features): best λ={best_lam_v9:.4f}, CV accuracy={acc_v9_best:.4f}")
print("(Book: MLR CV accuracy = 91.4%)")

# CV for k-NN on 9 features
k_vals_9 = np.arange(1, 31)
acc_knn9 = cv_knn_scan(Xv9, yv, Kv, k_vals_9)
best_k9  = k_vals_9[acc_knn9.argmax()]
print(f"k-NN (9 features): best k={best_k9}, CV accuracy={acc_knn9.max():.4f}")
print("(Book: k-NN best k=4, CV accuracy = 90.0%)")

---
## Exercises

### Exercise 1 — Log-odds interpretation (Book Exercise 11.7)

Starting from a prior probability of diabetes of 40%, compute the revised probability
if the log odds increase by:
(a) 1.08 (one standard deviation increase in fasting blood sugar);
(b) 0.037 (one standard deviation increase in cholesterol triglycerides).

Use the formula: if the prior log odds are $\ell_0 = \log(\pi/(1-\pi))$ and
the log odds increase by $\delta$, the posterior probability is $\sigma(\ell_0 + \delta)$.

In [ ]:
# Your answer here
pi_prior = 0.4
# log_odds_prior = ...
# revised_prob_a  = ...
# revised_prob_b  = ...

In [ ]:
# Solution
pi_prior = 0.4
lo0 = np.log(pi_prior / (1 - pi_prior))   # = log(0.4/0.6) ≈ -0.405

for delta, label in [(1.08, 'FBS (δ=1.08)'), (0.037, 'Chol (δ=0.037)')]:
    rev = sigmoid(lo0 + delta)
    print(f"{label}: revised P(diabetes) = {rev:.4f}")

### Exercise 2 — Baseline cross-entropy loss (Book Exercise 11.10)

A classifier that always predicts the training base rate $\hat\pi$ achieves a specific
cross-entropy loss.  Compute this for the diabetes dataset and verify it matches the
"uninformative predictor" limit seen in Figure 11.4.

In [ ]:
# Your answer here
# baseline_ce = ...

In [ ]:
# Solution
pi_hat = y.mean()
baseline_ce = -(pi_hat * np.log(pi_hat) + (1 - pi_hat) * np.log(1 - pi_hat))
print(f"Base rate π̂ = {pi_hat:.4f}")
print(f"Baseline CE loss = {baseline_ce:.4f}")
print("This matches the plateau seen in Figure 11.4(a) as λ → ∞.")

### Exercise 3 — The softmax reduces to sigmoid for K = 2

> **Book link — §11.5.1.**  Show algebraically, and verify numerically, that when $K=2$
> the multinomial logistic regression model is equivalent to binary logistic regression.

In [ ]:
# Numerical verification: fit both models on the diabetes dataset
# and compare predicted probabilities.

# Binary LR on full 5-predictor set
X5_s_full, _ = standardise(X5_raw, X5_raw)
b0_bin, beta_bin = fit_logreg(X5_s_full, y, lam=LAM_FINAL)

# Multinomial LR with K=2 on same data
b0_mn, B_mn = fit_softmax(X5_s_full, y.astype(int), K=2, lam=LAM_FINAL)

p_binary = predict_proba(X5_s_full, b0_bin, beta_bin)
p_multi  = predict_proba_softmax(X5_s_full, b0_mn, B_mn)[:, 1]

print(f"Max |p_binary - p_multi| = {np.abs(p_binary - p_multi).max():.6f}")
print("→ Both models assign (nearly) identical probabilities, confirming equivalence.")